In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("edgeR", "janitor", "tidyverse")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

### Load in Data

In [2]:
## load in raw counts
raw_reads <- read_tsv("results/abundances/stringtie_gene_counts.txt", skip = 1) %>%
    select(-c(Chr, Start, End, Strand, Length))

Rows: 16624 Columns: 230
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr   (5): Geneid, Chr, Start, End, Strand
dbl (225): Length, /home/clavery/projects/pse_ag/analysis/tau/rnaseq/HISAT-2...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
## load in SRA run table for sample names
raw_sra_run <- read_csv("SraRunTable.csv")

Rows: 1712 Columns: 34
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (27): Run, Assay Type, BioProject, BioSample, Center Name, Consent, DAT...
dbl   (5): AvgSpotLen, Bases, Bytes, replicate, version
dttm  (2): ReleaseDate, create_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Clean Up Data and Normalize

In [4]:
## clean up SRA run table
sra_run <- raw_sra_run %>%
    janitor::clean_names()

## create new names from SRA table
new_names <- sra_run %>%
    separate_wider_delim(create_date, delim = " ", names = c("date", "time")) %>%
    separate_wider_delim(date, delim = "-", names = c("year", "month", "day")) %>%
    mutate(organ = case_when(tissue == "abdomen without digestive or reproductive system" ~ "abdomen",
                             tissue == "digestive plus excretory system" ~ "digestive",
                             tissue == "gonad" ~ "gonad",
                             tissue == "head" ~ "head",
                             tissue == "reproductive system without gonad" ~ "rt",
                             tissue == "thorax without digestive system" ~ "thorax",
                             tissue == "whole body" ~ "body"),
           new = case_when(organism == "Drosophila pseudoobscura" ~ paste(year, sex, organ, replicate, sep = "_"),
                           TRUE ~ run)) %>%
    select(run, new) %>%
    rename(name = run)
    
## rename columns with sample info
names <- colnames(raw_reads) %>%
    as.data.frame() %>%
    rename(name = 1) %>%
    mutate(name = str_replace(name, "/home/clavery/projects/pse_ag/analysis/tau/rnaseq/HISAT-2/Dpse/", ""),
           name = str_replace(name, "_Dpse.csorted.hisat2.bam", "")) %>%
    left_join(new_names, by = "name") %>%
    pull("new")
names[1] <- "gene"
colnames(raw_reads) <- names

## select only Drosophila pseudoobscura reads
dpse_reads <- raw_reads %>%
    select(!contains("SRR")) %>%
    column_to_rownames("gene")

In [5]:
## create groups information
sample_info <- colnames(dpse_reads) %>%
    as.data.frame() %>%
    rename(sample = 1) %>%
    separate_wider_delim(sample, delim = "_", names = c("year", "sex", "organ", "replicate"), cols_remove = FALSE) %>%
    mutate(group = paste(organ, sex, sep = "_"))
groups <- sample_info$group

## create design matrix 
design <- model.matrix(~0 + groups)
rownames(design) <- colnames(dpse_reads)
colnames(design) <- str_replace(colnames(design), "groups", "")

head(design)

,abdomen_female,abdomen_male,body_female,body_male,digestive_female,digestive_male,gonad_female,gonad_male,head_female,head_male,rt_female,rt_male,thorax_female,thorax_male
2017_female_abdomen_1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2017_female_abdomen_2,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2017_female_abdomen_3,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2017_male_abdomen_1,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2017_male_abdomen_2,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2017_male_abdomen_3,0,1,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
## normalize count matrix with edger
dge_list <- DGEList(counts = dpse_reads, group = groups)
norm_list <- calcNormFactors(dge_list, method = "upperquartile")
disp_list <- estimateCommonDisp(norm_list, design, robust = T)

## get normalized counts
norm_counts <- disp_list$pseudo.counts %>%
    as.data.frame() %>%
    rownames_to_column("gene")

## get mean and log transform normalized counts
dpse_log <- norm_counts %>%
    pivot_longer(!gene, names_to = "sample", values_to = "value") %>%
    separate_wider_delim(sample, delim = "_", names = c("year", "sex", "organ", "replicate")) %>%
    mutate(tissue = paste(sex, organ, sep = "_")) %>%
    group_by(gene, tissue) %>%
    reframe(mean = log(mean(value))) %>%
    mutate(mean = case_when(mean < 0 ~ 0,
                            TRUE ~ mean)) %>%
    pivot_wider(names_from = tissue, values_from = mean) %>%
    column_to_rownames("gene")

### Calculate Tau

In [7]:
## create tau function
tau_function <- function(j){
	sum(1-dpse_log[j,]/max(dpse_log[j,]))/(length(dpse_log[j,])-1)
	}

## apply to matrix
tau_est <- sapply(1:nrow(dpse_log), tau_function)

## save tau as a data frame
tau_df <- tau_est %>%
    as.data.frame() %>%
    rename("tau" = 1) %>%
    mutate(tau = replace_na(tau, 0),
           gene = rownames(dpse_log)) %>%
    select(gene, tau)
head(tau_df)

,gene,tau
,<chr>,<dbl>
1,ATP6,0.1851908
2,ATP8,0.5663132
3,COX1,0.1480673
4,COX2,0.2524231
5,COX3,0.1500493
6,CYTB,0.2038775


In [8]:
## find the difference between tissues and assign tissue specificity
dpse_tau <- dpse_log %>%
    rownames_to_column("gene") %>%
    left_join(tau_df, by = "gene") %>%
    pivot_longer(!c(gene, tau), names_to = "sample", values_to = "value") %>%
    group_by(gene, tau) %>%
    reframe(max_organ = sample[which.max(value)],
            max = max(value),
            max2 = sort(value, decreasing = TRUE)[2],
            perc = (max-max2)/max) %>%
    select(-c(max, max2)) %>%
    mutate(perc = replace_na(perc, 0),
           tau_tissue = case_when(perc > 0.1 & tau > 0.5 ~ max_organ,
                                  TRUE ~ "ambiguous"))
head(dpse_tau)

gene,tau,max_organ,perc,tau_tissue
<chr>,<dbl>,<chr>,<dbl>,<chr>
ATP6,0.1851908,male_thorax,0.026911706,ambiguous
ATP8,0.5663132,male_thorax,0.125303784,male_thorax
COX1,0.1480673,male_thorax,0.007098942,ambiguous
COX2,0.2524231,male_thorax,0.016269619,ambiguous
COX3,0.1500493,male_thorax,0.028558177,ambiguous
CYTB,0.2038775,male_thorax,0.045447648,ambiguous


In [9]:
## find the percent of genes in each tissue specific
dpse_tau %>%
    group_by(tau_tissue) %>%
    reframe(count = n()) %>%
    mutate(percent = (count/sum(count))*100)

tau_tissue,count,percent
<chr>,<int>,<dbl>
ambiguous,11974,72.02839269
female_abdomen,143,0.86020212
female_body,12,0.07218479
female_digestive,180,1.08277190
female_gonad,229,1.37752647
female_head,110,0.66169394
female_rt,140,0.84215592
female_thorax,78,0.46920115
male_abdomen,77,0.46318576


In [10]:
## write csv
write_csv(dpse_tau, "results/tau_tissue_0.50.csv")